# Assignment 04: Blood Pressure Follow-Up

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

DATA_PATH = Path("data") / "bp_followup.csv"
HOME_PATH = Path("data") / "home_bp.csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
LOADED_PATH = OUTPUT_DIR / "bp_loaded.csv"
SUMMARY_PATH = OUTPUT_DIR / "visit_summary.csv"
COUNTS_PATH = OUTPUT_DIR / "clinic_counts.csv"
FOLLOWUP_PATH = OUTPUT_DIR / "followup_priority.csv"
PARQUET_PATH = OUTPUT_DIR / "followup_priority.parquet"
GAP_PATH = OUTPUT_DIR / "white_coat_gap.csv"

print("data files found:", DATA_PATH.exists() and HOME_PATH.exists())

NumPy: 2.3.3
pandas: 3.0.5
pyarrow: 25.0.0
data files found: True


## Task 1: Put the cells in running order

### 1.1 Reorder the cells

The next two code cells are out of order: the first uses `vials_on_hand`, which the second defines. Move the whole producer cell above the dependent cell. Do not copy the definition into another cell.

In [14]:
vials_on_hand = 3
print("vials_on_hand:", vials_on_hand)

vials_on_hand: 3


In [15]:
doses_available = vials_on_hand * 10
print("doses_available:", doses_available)

doses_available: 30


### 1.2 Explain the repair

Explain the difference between the order the cells appear in and the order they ran, why the first cell can print a result even while it sits above the cell it needs, why output saved under a cell is not proof that the notebook works now, and what you changed and how Restart and Run All shows that it worked.

The order in which cells appear can differ from the order that the user runs them. A dependent cell can print a result above the cell it needs information from if the cell below it has been run previously and the output variables have been stored in the kernel. Output saved under a cell is just from the last time it ran so if it has been edited since, there is no evidence that it works now. I changed the order so that vials on hand was defined before it was called in the doses available calculation. Restart and run all deletes the existing variables and runs all the code so doing this confirms that the code as written at that time all works. periwinkle.

## Task 2: Load the blood pressure export

### 2.1 Look at the raw file

Run this cell as it is: it prints the file and reads it with no options.

In [16]:
print(open(DATA_PATH).read())
raw = pd.read_csv(DATA_PATH)
print("shape:", raw.shape)
raw.head(3)

patient_id;clinic;age;sbp_baseline;sbp_week4;sbp_week8;coordinator_note
id;text;years;mmHg;mmHg;mmHg;text
P101;North;58;152;146;138;Started lisinopril at baseline
P102;South;64;146;140;135;
P103;West;47;138;134;131;Diet counseling only
P104;North;71;164;-999;150;Missed week 4 visit
P105;West;55;158;150;147;
P106;East;62;149;141;136;Dose increased at week 4
P107;North;49;140;137;133;
P108;South;68;171;160;152;Referred to cardiology
P109;West;60;155;149;-999;Moved before week 8
P110;East;53;160;152;145;
P111;East;66;-999;145;139;Baseline cuff too small
P112;North;45;132;130;128;
P113;East;70;168;158;155;Home monitor issued
P114;North;57;145;143;137;

shape: (15, 1)


,patient_id;clinic;age;sbp_baseline;sbp_week4;sbp_week8;coordinator_note
0,id;text;years;mmHg;mmHg;mmHg;text
1,P101;North;58;152;146;138;Started lisinopril a...
2,P102;South;64;146;140;135;


### 2.2 Read with options and save

> **Checkpoint: `output/bp_loaded.csv`**
> 14 rows under the header `patient_id,clinic,age,sbp_baseline,sbp_week4,sbp_week8`.

In [17]:
bp = pd.read_csv(DATA_PATH, sep = ";", skiprows = [1], usecols = ["patient_id", "clinic", "age", "sbp_baseline", "sbp_week4", "sbp_week8"], na_values = ["-999"], index_col = "patient_id")  #reads DATA_PATH with sep, skiprows, usecols, na_values, and index_col

print("shape:", bp.shape)
print(bp.dtypes)
display(bp.head())

#display patient P104's row with .loc, then the first three rows with .iloc
display(bp.loc["P104"])
display(bp.iloc[0:3])
#save bp to LOADED_PATH, keeping its patient_id index
bp.to_csv(LOADED_PATH, index=True)

shape: (14, 5)
clinic              str
age               int64
sbp_baseline    float64
sbp_week4       float64
sbp_week8       float64
dtype: object


,clinic,age,sbp_baseline,sbp_week4,sbp_week8
patient_id,,,,,
P101,North,58,152.0,146.0,138.0
P102,South,64,146.0,140.0,135.0
P103,West,47,138.0,134.0,131.0
P104,North,71,164.0,NaN,150.0
P105,West,55,158.0,150.0,147.0


clinic          North
age                71
sbp_baseline    164.0
sbp_week4         NaN
sbp_week8       150.0
Name: P104, dtype: object

,clinic,age,sbp_baseline,sbp_week4,sbp_week8
patient_id,,,,,
P101,North,58,152.0,146.0,138.0
P102,South,64,146.0,140.0,135.0
P103,West,47,138.0,134.0,131.0


## Task 3: Summarize the readings

### 3.1 Summarize each visit

> **Checkpoint: `output/visit_summary.csv`**
> Three rows, `sbp_baseline`, `sbp_week4`, and `sbp_week8`, under the header `visit,mean,median,count`.

In [18]:
readings = bp[["sbp_baseline", "sbp_week4", "sbp_week8"]]

visit_summary = pd.DataFrame({
    "mean": readings.mean(),
    "median": readings.median(),
    "count": readings.count()
})  #a DataFrame with the columns mean, median, and count, one row per visit
visit_summary.index.name = "visit" #name the index visit

display(visit_summary)

highest_baseline = readings["sbp_baseline"].idxmax()  #the patient_id with the highest sbp_baseline
baseline_week8_r = readings["sbp_baseline"].corr(readings["sbp_week8"])  #the correlation of sbp_baseline with sbp_week8
print("highest baseline:", highest_baseline) 
print("baseline vs week 8 r:", baseline_week8_r) 

#save visit_summary to SUMMARY_PATH, keeping its visit index
visit_summary.to_csv(SUMMARY_PATH, index=True)

,mean,median,count
visit,,,
sbp_baseline,152.153846,152.0,13
sbp_week4,145.000000,145.0,13
sbp_week8,140.461538,138.0,13


highest baseline: P108
baseline vs week 8 r: 0.9764184782678085


### 3.2 Count patients per clinic

> **Checkpoint: `output/clinic_counts.csv`**
> Four rows under the header `clinic,count`, North first.

In [19]:
clinic_counts = bp["clinic"].value_counts()  #the number of patients at each clinic, with value_counts()
display(clinic_counts)
print("clinics:", bp["clinic"].nunique())  #the number of distinct clinics, from nunique()

#save clinic_counts to COUNTS_PATH
clinic_counts.to_csv(COUNTS_PATH, index=True)

clinic
North    5
East     4
West     3
South    2
Name: count, dtype: int64

clinics: 4


## Task 4: Build the follow-up list

### 4.1 Select the program patients

In [20]:
in_program = (bp["clinic"].isin(["North", "East"])) & (bp["sbp_baseline"] >= 140)  # TODO: True for North or East patients (isin) whose sbp_baseline is 140 or more
followup = bp.loc[in_program].drop(columns = ["age"])  #the in_program rows of bp, without the age column

print("program patients:", in_program.sum())
followup

program patients: 7


,clinic,sbp_baseline,sbp_week4,sbp_week8
patient_id,,,,
P101,North,152.0,146.0,138.0
P104,North,164.0,NaN,150.0
P106,East,149.0,141.0,136.0
P107,North,140.0,137.0,133.0
P110,East,160.0,152.0,145.0
P113,East,168.0,158.0,155.0
P114,North,145.0,143.0,137.0


### 4.2 Add the derived columns

In [21]:
followup["sbp_mean"] = followup[["sbp_baseline", "sbp_week4", "sbp_week8"]].mean(axis="columns")  #add sbp_mean, the mean of each patient's three readings (axis="columns")
followup["change_week8"] = followup["sbp_week8"] - followup["sbp_baseline"]  #add change_week8, sbp_week8 minus sbp_baseline

followup

,clinic,sbp_baseline,sbp_week4,sbp_week8,sbp_mean,change_week8
patient_id,,,,,,
P101,North,152.0,146.0,138.0,145.333333,-14.0
P104,North,164.0,NaN,150.0,157.000000,-14.0
P106,East,149.0,141.0,136.0,142.000000,-13.0
P107,North,140.0,137.0,133.0,136.666667,-7.0
P110,East,160.0,152.0,145.0,152.333333,-15.0
P113,East,168.0,158.0,155.0,160.333333,-13.0
P114,North,145.0,143.0,137.0,141.666667,-8.0


### 4.3 Sort, rank, and save

> **Checkpoint: `output/followup_priority.csv` and `output/followup_priority.parquet`**
> Seven patients, P110 first and P107 last, with the columns `README.md` lists.

In [22]:
followup = followup.sort_values(by = ["change_week8", "patient_id"], ascending=[True, True])  #sort followup by change_week8, then patient_id, and assign the result back to followup
followup["improvement_rank"] = followup["change_week8"].rank(method="min")  #add improvement_rank, the rank of change_week8 with method="min"
display(followup)

#save followup to FOLLOWUP_PATH, keeping its patient_id index
followup.to_csv(FOLLOWUP_PATH, index=True)
#save followup to PARQUET_PATH with to_parquet(), keeping its index (no index=False)
followup.to_parquet(PARQUET_PATH, index=True)

back = pd.read_parquet(PARQUET_PATH)  #read PARQUET_PATH back with pd.read_parquet()
print("Parquet round trip matches:", back.equals(followup))

,clinic,sbp_baseline,sbp_week4,sbp_week8,sbp_mean,change_week8,improvement_rank
patient_id,,,,,,,
P110,East,160.0,152.0,145.0,152.333333,-15.0,1.0
P101,North,152.0,146.0,138.0,145.333333,-14.0,2.0
P104,North,164.0,NaN,150.0,157.000000,-14.0,2.0
P106,East,149.0,141.0,136.0,142.000000,-13.0,4.0
P113,East,168.0,158.0,155.0,160.333333,-13.0,4.0
P114,North,145.0,143.0,137.0,141.666667,-8.0,6.0
P107,North,140.0,137.0,133.0,136.666667,-7.0,7.0


Parquet round trip matches: True


## Task 5: Compare clinic and home readings

> **Checkpoint: `output/white_coat_gap.csv`**
> 15 rows under the header `patient_id,white_coat_gap_mmhg`, five of them with a number.

In [ ]:
home = pd.read_csv(HOME_PATH, index_col="patient_id")  #read HOME_PATH with patient_id as the index

white_coat_gap = bp["sbp_week8"] - home["home_sbp_week8"]  #bp's sbp_week8 minus home's home_sbp_week8
white_coat_gap.name = "white_coat_gap_mmhg"  #name the Series white_coat_gap_mmhg

display(white_coat_gap)
print("patients with both readings:", white_coat_gap.count())

#save white_coat_gap to GAP_PATH, keeping its patient_id index
white_coat_gap.to_csv(GAP_PATH, index=True)

patient_id
P101    7.0
P102    NaN
P103    NaN
P104    4.0
P105    NaN
P106    3.0
P107    NaN
P108    NaN
P109    NaN
P110    5.0
P111    NaN
P112    NaN
P113    7.0
P114    NaN
P115    NaN
Name: white_coat_gap_mmhg, dtype: float64

patients with both readings: 5


## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [24]:
from check_assignment import run_checks

run_checks()

Checks: latest from the course repository (main).
[PASS]  4/4  notebook: Task 1.2 explanation
[PASS]  4/4  bp loaded: patient_id column
[PASS]  4/4  bp loaded: units row skipped
[PASS]  4/4  bp loaded: coordinator_note left out
[PASS]  4/4  bp loaded: all 14 patients once
[PASS]  4/4  bp loaded: -999 read as missing
[PASS]  4/4  bp loaded: clinic, age, and readings
[PASS]  3/3  visit summary: one row per visit
[PASS]  5/5  visit summary: mean
[PASS]  5/5  visit summary: median
[PASS]  5/5  visit summary: count
[PASS]  4/4  clinic counts: patients per clinic
[PASS]  2/2  clinic counts: most common first
[PASS]  3/3  follow-up list: patient_id column
[PASS]  6/6  follow-up list: program patients
[PASS]  2/2  follow-up list: derived column names
[PASS]  2/2  follow-up list: age dropped
[PASS]  4/4  follow-up list: sbp_mean values
[PASS]  4/4  follow-up list: change_week8 values
[PASS]  2/2  follow-up list: improvement_rank values
[PASS]  3/3  follow-up list: largest drop first
[PASS]  2/2

{'schema': 'datasci217/grading-result/v1',
 'score': 100,
 'max-score': 100,
 'tests': [{'test-name': 'notebook: Task 1.2 explanation',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: patient_id column',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: units row skipped',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: coordinator_note left out',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: all 14 patients once',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: -999 read as missing',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'bp loaded: clinic, age, and readings',
   'passed': True,
   'score': 4,
   'max-score': 4,
   'detail': ''},
  {'test-name': 'visit summary: one row per visit',
   'p